# Llama 3 8B Original vs Compressed Size

This notebook loads the local `Meta-Llama-3-8B` model and computes the total storage size of the original and compressed versions.

Assumptions:
- The original model is stored in 16-bit precision.
- `model.embed_tokens` and `lm_head` stay in 16-bit precision.
- All other parameters are stored with a user-provided `bits_per_parameter`.
- Parameter counts are computed from the loaded model, not from hard-coded architecture constants.

Edit `bits_per_parameter_values` in the last cell to evaluate any compression point you want.


In [2]:
from pathlib import Path

from transformers import AutoModelForCausalLM


CANDIDATE_MODEL_PATHS = [
    "/workspace/Weight_compression/Wparam_dataset/hf_model/meta-llama--Meta-Llama-3-8B",
    "/home/jgryu/workspace/weight_compression/Wparam_dataset/hf_model/meta-llama--Meta-Llama-3-8B",
]

ORIGINAL_BITS = 16
FIXED_BITS = 16


def human_size(num_bytes, base=1024):
    units = ["B", "KiB", "MiB", "GiB", "TiB"] if base == 1024 else ["B", "KB", "MB", "GB", "TB"]
    value = float(num_bytes)
    for unit in units:
        if value < base or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= base


def format_bytes(num_bytes):
    return f"{num_bytes:,.0f} B ({human_size(num_bytes, 1000)}, {human_size(num_bytes, 1024)})"


def resolve_model_path(candidate_paths):
    for candidate in candidate_paths:
        path = Path(candidate)
        if path.exists():
            return str(path)
    raise FileNotFoundError(
        "Could not find Meta-Llama-3-8B model path. Checked: " + ", ".join(candidate_paths)
    )


MODEL_PATH = resolve_model_path(CANDIDATE_MODEL_PATHS)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype="auto",
    low_cpu_mem_usage=True,
    local_files_only=True,
)
model.eval()

print(f"Loaded model from: {MODEL_PATH}")


/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/opt/conda/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading checkpoint shards: 100%|██████████| 7/7 [00:00<00:00, 10.34it/s]

Loaded model from: /home/jgryu/workspace/weight_compression/Wparam_dataset/hf_model/meta-llama--Meta-Llama-3-8B


In [3]:
FIXED_PREFIXES = ("model.embed_tokens.", "lm_head.")


def count_parameter_groups(model):
    fixed_param_counts = {}
    total_params = 0
    fixed_params = 0

    for name, param in model.named_parameters():
        num_params = param.numel()
        total_params += num_params
        if name.startswith(FIXED_PREFIXES):
            fixed_params += num_params
            fixed_param_counts[name] = num_params

    variable_params = total_params - fixed_params
    return {
        "total_params": total_params,
        "fixed_params": fixed_params,
        "variable_params": variable_params,
        "fixed_param_counts": fixed_param_counts,
    }


COUNTS = count_parameter_groups(model)

summary_rows = [
    ("Embedding params", COUNTS["fixed_param_counts"].get("model.embed_tokens.weight", 0)),
    ("LM head params", COUNTS["fixed_param_counts"].get("lm_head.weight", 0)),
    ("Fixed 16-bit params", COUNTS["fixed_params"]),
    ("Other params", COUNTS["variable_params"]),
    ("Total params", COUNTS["total_params"]),
]

print(f"Model name: {model.config._name_or_path}")
print()
for label, value in summary_rows:
    print(f"{label:<20}: {value:>15,}")

original_bytes = COUNTS["total_params"] * ORIGINAL_BITS / 8
fixed_16bit_bytes = COUNTS["fixed_params"] * FIXED_BITS / 8

print()
print(f"Original 16-bit total size            : {format_bytes(original_bytes)}")
print(f"Fixed 16-bit size in compressed model : {format_bytes(fixed_16bit_bytes)}")


Model name: /home/jgryu/workspace/weight_compression/Wparam_dataset/hf_model/meta-llama--Meta-Llama-3-8B

Embedding params    :     525,336,576
LM head params      :     525,336,576
Fixed 16-bit params :   1,050,673,152
Other params        :   6,979,588,096
Total params        :   8,030,261,248

Original 16-bit total size            : 16,060,522,496 B (16.06 GB, 14.96 GiB)
Fixed 16-bit size in compressed model : 2,101,346,304 B (2.10 GB, 1.96 GiB)


In [4]:
def calculate_sizes(bits_per_parameter, counts=COUNTS):
    original_bytes = counts["total_params"] * ORIGINAL_BITS / 8
    compressed_bytes = (
        counts["fixed_params"] * FIXED_BITS / 8
        + counts["variable_params"] * bits_per_parameter / 8
    )
    savings_bytes = original_bytes - compressed_bytes
    return {
        "bits_per_parameter": bits_per_parameter,
        "original_bytes": original_bytes,
        "compressed_bytes": compressed_bytes,
        "savings_bytes": savings_bytes,
        "compression_ratio": original_bytes / compressed_bytes,
        "relative_size_percent": 100 * compressed_bytes / original_bytes,
    }


In [5]:
import argparse
import os
import time

import glog, json

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

import torch
import torch.multiprocessing as mp
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.modeling_attn_mask_utils import \
    _prepare_4d_causal_attention_mask


from operator import attrgetter

from pathlib import Path
import sys

candidate_project_roots = [
    Path.cwd(),
    Path.cwd() / 'workspace/weight_compression',
    Path('/workspace/Weight_compression'),
    Path('/home/jgryu/workspace/weight_compression'),
]

project_root = None
for candidate in candidate_project_roots:
    if (candidate / 'NWC').exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError('Could not resolve project root containing the NWC package.')

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from NWC.models import get_model

device = torch.device('cpu')

class Config:
    def __init__(self, **entries):
        self.__dict__.update(entries)

# comp_model_path = '../NWC/checkpoint/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/M16/lmbda50_rdloss_ql_size16_encdim512_M16_Q4_R0_m0_batch_size2048_total_iter200000_lr0.0001_seed100/best_loss_model_loss_3.87239_bpp_4.65884_MSE_0.0162_total_iter_95000.pth.tar'
# comp_model_path = '/home/jgryu/workspace/weight_compression/NWC/checkpoint/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/M16/lmbda10000_rdloss_ql_size16_encdim512_M16_Q4_R0_m0_batch_size2048_total_iter200000_lr0.0001_seed100/best_loss_model_loss_11.43015_bpp_6.30844_MSE_0.00043_total_iter_200000.pth.tar'
# comp_model_path = '/workspace/Weight_compression/NWC/checkpoint/nwc_scale_cond/block_seq_scale_cond_scaler_meta-llama--Meta-Llama-3-8B__scaleH_sig0.0001_std_rnormed_with_col_std_lidx_row_1024.pt/rdloss_size128_encdim1024_M256_Q0_R0_m0_batch_size2048_total_iter200000_lr0.0001_seed100/lmbda50_/best_loss_model_loss_3.94749_bpp_3.26997_MSE_4.91093_total_iter_192500.pth.tar'
comp_model_path = '/home/jgryu/workspace/weight_compression/NWC/checkpoint/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/M16/lmbda300_rdloss_ql_size16_encdim512_M16_Q4_R0_m0_batch_size2048_total_iter200000_lr0.0001_seed100/best_loss_model_loss_5.32101_bpp_5.72603_MSE_0.00289_total_iter_95000.pth.tar'  ## 3.95 bits
# comp_model_path = '/home/jgryu/workspace/weight_compression/NWC/checkpoint/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/M16/lmbda75_/best_loss_model_loss_4.18422_bpp_4.90131_MSE_0.01087_total_iter_47500.pth.tar' # 2.96 bit
# comp_model_path = "/home/jgryu/workspace/weight_compression/NWC/checkpoint/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/M16/lmbda30_rdloss_ql_size16_encdim512_M16_Q4_R0_m0_batch_size2048_total_iter200000_lr0.0001_seed100/best_loss_model_loss_3.46368_bpp_4.27494_MSE_0.02685_total_iter_95000.pth.tar" # 2.31 bit
# comp_model_path = "/home/jgryu/workspace/weight_compression/NWC/checkpoint2/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/rdloss_ql_size16_encdim512_M16_Q4_R0_m0_batch_size2048_total_iter200000_lr0.0001_seed100/n_rb1/lmbda300_/best_loss_model_loss_5.27489_bpp_5.82522_MSE_0.0027_total_iter_47500.pth.tar" ## nres1 4bit
# comp_model_path = "/home/jgryu/workspace/weight_compression/NWC/checkpoint2/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/rdloss_ql_size16_encdim512_M16_Q4_R0_m0_batch_size2048_total_iter200000_lr0.0001_seed100/n_rb2/lmbda300_/best_loss_model_loss_5.2877_bpp_5.76578_MSE_0.00275_total_iter_47500.pth.tar"  ## nres2 4bit
# comp_model_path = "/home/jgryu/workspace/weight_compression/NWC/checkpoint2/nwc_ql/block_seq_ql_random_scaler_meta-llama--Meta-Llama-3-8B__col_1024_gaussian_padding.pt/rdloss_ql_size4_encdim64_M4_Q4_R0_m0_batch_size8192_total_iter200000_lr0.0001_seed100/lmbda300_/best_loss_model_loss_5.11241_bpp_5.99794_MSE_0.00262_total_iter_82500.pth.tar"   ## nres2 size 4  4bit

config = os.path.join(os.path.dirname(comp_model_path), 'config.json')
with open(config, 'r', encoding='utf-8') as file:
    config = json.load(file)
config = Config(**config)

shift, scale = None, None
if config.architecture == 'nwc_ql' and not hasattr(config, "Q"):
    config.Q = 4
if not hasattr(config, "no_layernorm"):
    config.no_layernorm = False


comp_model = get_model(config.architecture, config, scale=scale, shift=shift)
comp_model.config = config
ckpt = torch.load(comp_model_path, weights_only=False, map_location = device)
scale, shift  = torch.zeros(1), torch.zeros(1)

comp_model.load_state_dict(ckpt["state_dict"], strict = False)
comp_model.scale = scale.to(device)
comp_model.shift = shift.to(device)

comp_model.to(device)
comp_model.eval()
comp_model.update()

comp_model.update(force=True)              # CompressAI: CDF 고정 및 버퍼 등록
# comp_model.entropy_bottleneck._quantized_cdf  # 캐시되어 이후 재계산 안 됨


import torch
from compressai.entropy_models import EntropyBottleneck

# 예시: 채널 수가 192인 EntropyBottleneck 인스턴스 생성
entropy_bottleneck = comp_model.entropy_bottleneck
cdf_size = entropy_bottleneck._quantized_cdf.numel()
length_size = entropy_bottleneck._cdf_length.numel()
offset_size = entropy_bottleneck._offset.numel()


def count_params(module, trainable_only=False):
    if module is None:
        return 0
    if trainable_only:
        return sum(p.numel() for p in module.parameters() if p.requires_grad)
    return sum(p.numel() for p in module.parameters())

# g_s 파라미터 개수
gs_params = count_params(comp_model.g_s)
print(f"g_s 파라미터 개수: {gs_params:,}")

quality_emb_params = count_params(comp_model.quality_embedding)
print(f"quality_embedding 파라미터 개수: {quality_emb_params:,}")

print(f"메인 코드북(_quantized_cdf) 요소 개수: {cdf_size:,}")
print(f"CDF 길이 정보(_cdf_length) 요소 개수: {length_size:,}")
print(f"오프셋 정보(_offset) 요소 개수: {offset_size:,}")

# entropy bottleneck 쪽 디코딩 메타데이터
total_decoding_params = cdf_size + length_size + offset_size + quality_emb_params + gs_params
print(f"디코딩용 총 요소 개수: {total_decoding_params:,}")

I0326 17:22:47.186725 1896310 utils.py:148] Note: detected 128 virtual cores but NumExpr set to maximum of 64, check "NUMEXPR_MAX_THREADS" environment variable.
I0326 17:22:47.187993 1896310 utils.py:151] Note: NumExpr detected 128 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
I0326 17:22:47.188853 1896310 utils.py:164] NumExpr defaulting to 16 threads.
W0326 17:22:47.540412 1896310 warnings.py:109] /opt/conda/lib/python3.10/site-packages/compressai/models/video/google.py:353: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)



g_s 파라미터 개수: 1,071,632
quality_embedding 파라미터 개수: 2,048
메인 코드북(_quantized_cdf) 요소 개수: 3,440
CDF 길이 정보(_cdf_length) 요소 개수: 16
오프셋 정보(_offset) 요소 개수: 16
디코딩용 총 요소 개수: 1,077,152


In [9]:

if 'total_decoding_params' not in globals():
    raise NameError('Run the comp_model cell first so total_decoding_params is defined.')

bits_per_parameter_values = [3.16, 4.04, 5.00, 5.98]
comp_model_decoding_bytes = total_decoding_params * FIXED_BITS / 8

print(f"Original 16-bit total size: {format_bytes(COUNTS['total_params'] * ORIGINAL_BITS / 8)}")
print(f"comp_model decoding size (16-bit): {format_bytes(comp_model_decoding_bytes)}")
print()

header = (
    f"{'Other bpp':>9} | {'Total compressed size':>33} | {'Saved size':>33} | {'Compression':>11} | {'Relative size':>13} | {'comp_model share':>16}"
)
print(header)
print('-' * len(header))

for bits_per_parameter in bits_per_parameter_values:
    result = calculate_sizes(bits_per_parameter)
    total_compressed_bytes = result['compressed_bytes'] + comp_model_decoding_bytes
    total_savings_bytes = result['original_bytes'] - total_compressed_bytes
    total_compression_ratio = result['original_bytes'] / total_compressed_bytes
    total_relative_size_percent = 100 * total_compressed_bytes / result['original_bytes']
    comp_model_share_percent = 100 * comp_model_decoding_bytes / total_compressed_bytes
    print(
        f"{bits_per_parameter:9.2f} | "
        f"{human_size(total_compressed_bytes, 1000):>15} / {human_size(total_compressed_bytes, 1024):>14} | "
        f"{human_size(total_savings_bytes, 1000):>15} / {human_size(total_savings_bytes, 1024):>14} | "
        f"{total_compression_ratio:10.2f}x | "
        f"{total_relative_size_percent:12.2f}% | "
        f"{comp_model_share_percent:15.2f}%"
    )


Original 16-bit total size: 16,060,522,496 B (16.06 GB, 14.96 GiB)
comp_model decoding size (16-bit): 2,154,304 B (2.15 MB, 2.05 MiB)

Other bpp |             Total compressed size |                        Saved size | Compression | Relative size | comp_model share
----------------------------------------------------------------------------------------------------------------------------------
     3.16 |         4.86 GB /       4.53 GiB |        11.20 GB /      10.43 GiB |       3.30x |        30.26% |            0.04%
     4.04 |         5.63 GB /       5.24 GiB |        10.43 GB /       9.72 GiB |       2.85x |        35.04% |            0.04%
     5.00 |         6.47 GB /       6.02 GiB |         9.59 GB /       8.94 GiB |       2.48x |        40.26% |            0.03%
     5.98 |         7.32 GB /       6.82 GiB |         8.74 GB /       8.14 GiB |       2.19x |        45.58% |            0.03%
